# R3-N1: selected LR calibration and development handoff

This notebook exports Protocol B LR validation predictions, fits temperature only on calibration recordings, evaluates the frozen fit and selects a threshold on separate operating recordings, then creates and verifies a new development bundle. The model is the selected 67-feature R2-M1 LR; the experimental M2 subset is not used. No existing bundle or lock is changed. Operating-selected metrics are development results, not independent test estimates.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
FEATURES = ROOT / 'data/features/r2_s1_B'
RUN = ROOT / 'runs/r3/r3_n1_lr/selected_lr_v1'
EXPORT, FIT, OPERATING, HANDOFF = [RUN / n for n in ('export', 'fit', 'operating', 'handoff')]
if not ROOT.is_dir():
    raise FileNotFoundError(f'Project folder unavailable: {ROOT}. Check the Drive mount and ROOT path.')
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

required = [ROOT/'configs/r3_n1_lr_contract.json', FEATURES/'r2_s1_manifest.json',
            FEATURES/'lr_train_crops_B_v1.npz', FEATURES/'lr_validation_B_v1.npz',
            ROOT/'runs/r3/r3_m2_adapters_verified_v3/lr/bundle.json',
            ROOT/'runs/r2/r2_m1_20261106T140611Z/evidence.json']
required += [ROOT / p for p in (
    'scripts/r3_n1_lr_export.py', 'scripts/r3_n1_lr_fit.py', 'scripts/r3_n1_lr_operating.py',
    'scripts/r3_n1_lr_bundle.py', 'scripts/r2_m1_lr.py',
    'src/echoproof/inference.py', 'src/echoproof/models/r3_n1_lr_bundle.py',
    'tests/test_r3_n1_lr_export.py', 'tests/test_r3_n1_lr_fit.py',
    'tests/test_r3_n1_lr_operating.py', 'tests/test_r3_n1_lr_bundle.py', 'tests/test_r2_bundle.py',
    'runs/r2/r2_m1_20261106T140611Z/model/weights.npz',
    'runs/r2/r2_m1_20261106T140611Z/model/lr_model.json',
    'runs/r2/r2_m1_20261106T140611Z/predictions_chunks.csv',
)]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing required Drive files:\n' + '\n'.join(missing))
for package in ('numpy', 'scipy', 'sklearn', 'librosa', 'soundfile', 'pyarrow', 'matplotlib', 'pytest'):
    __import__(package)

def run_command(args):
    if not ROOT.is_dir():
        raise FileNotFoundError(f'Project folder unavailable: {ROOT}. Check the Drive mount before continuing.')
    tail = []
    with subprocess.Popen([sys.executable, '-u', *map(str, args)], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=str(ROOT)) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail = (tail + [line.rstrip()])[-30:]
        status = process.wait()
    if status:
        raise RuntimeError('Command failed:\n' + '\n'.join(tail))

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_outputs(directory, evidence):
    outputs = evidence.get('outputs')
    if not isinstance(outputs, dict) or not outputs:
        raise ValueError(f'Missing artifact hashes in {directory}')
    for name, expected in outputs.items():
        path = (directory / name).resolve()
        if not path.is_relative_to(directory.resolve()) or not path.is_file() or digest(path) != expected:
            raise ValueError(f'Artifact missing, unsafe, or changed: {name}')

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

contract_path = ROOT / 'configs/r3_n1_lr_contract.json'
contract = read_json(contract_path)
print('Run location:', RUN)
print('Contract SHA-256:', digest(contract_path))


## Fixture and regression checks
These use synthetic data and disposable bundles. A test report is written to a temporary directory, leaving previous evidence untouched.


In [ ]:
import tempfile
import xml.etree.ElementTree as ET
with tempfile.TemporaryDirectory(prefix='r3_n1_lr_tests_') as temp:
    junit = Path(temp) / 'tests.xml'
    run_command(['-m', 'pytest', '-q', '-p', 'no:cacheprovider',
                 'tests/test_r3_n1_lr_export.py', 'tests/test_r3_n1_lr_fit.py',
                 'tests/test_r3_n1_lr_operating.py', 'tests/test_r3_n1_lr_bundle.py',
                 'tests/test_r2_bundle.py', '--junitxml', junit])
    suites = ET.parse(junit).getroot().findall('.//testsuite')
    assert suites and all(int(s.get(k, 0)) == 0 for s in suites for k in ('failures', 'errors', 'skipped'))


## 1. Export frozen LR scores
The exporter verifies model and feature hashes, role separation and selection-score parity. If a stage directory exists without complete evidence, stop and inspect it instead of overwriting it.


In [ ]:
if not EXPORT.exists():
    run_command([ROOT/'scripts/r3_n1_lr_export.py', '--feature-dir', FEATURES, '--output', EXPORT])
export = read_json(EXPORT/'evidence.json')
assert export['task'] == 'R3-N1-LR prediction export' and export['status'] == 'complete'
assert export['contract_sha256'] == digest(contract_path)
assert export['selected_bundle_sha256'] == contract['bundle_sha256']
verify_outputs(EXPORT, export)
print('Selection parity maximum difference:', export['selection_max_abs_difference'])
print('Export outputs:', sorted(export['outputs']))


## 2. Fit temperature on calibration only
This stage reads the calibration recording predictions, never the operating prediction file. Its fit metrics are in-sample diagnostics; the operating stage determines the declared acceptance rule.


In [ ]:
if not FIT.exists():
    run_command([ROOT/'scripts/r3_n1_lr_fit.py', '--export-dir', EXPORT, '--output', FIT])
fit = read_json(FIT/'temperature.json')
assert fit['task'] == 'R3-N1-LR temperature fit' and fit['status'] == 'calibration_fit_complete'
assert fit['source_contract_sha256'] == digest(contract_path)
assert fit['source_export_evidence_sha256'] == digest(EXPORT/'evidence.json')
assert fit['source_calibration_predictions_sha256'] == digest(EXPORT/'predictions_recordings_calibration.parquet')
assert fit['selected_bundle_sha256'] == contract['bundle_sha256'] and fit['fit_role'] == 'calibration'
print('Temperature:', fit['temperature'], '| calibration recordings:', fit['fit_recordings'])


## 3. Evaluate on operating and choose a threshold
Compare raw and temperature scores with the predeclared log-loss/Brier rule. Select the threshold using the project's macro-F1 tie-breaks. The resulting operating metrics are selected on this same partition.


In [ ]:
if not OPERATING.exists():
    run_command([ROOT/'scripts/r3_n1_lr_operating.py', '--export-dir', EXPORT, '--fit-dir', FIT, '--output', OPERATING])
operating = read_json(OPERATING/'evidence.json')
assert operating['task'] == 'R3-N1-LR operating evaluation' and operating['status'] == 'complete'
assert operating['source_contract_sha256'] == digest(contract_path)
assert operating['source_export_evidence_sha256'] == digest(EXPORT/'evidence.json')
assert operating['source_fit_sha256'] == digest(FIT/'temperature.json')
assert operating['source_operating_predictions_sha256'] == digest(EXPORT/'predictions_recordings_operating.parquet')
verify_outputs(OPERATING, operating)
print(json.dumps(operating, indent=2))


## 4. Package and verify the LR development handoff
This creates a new development bundle with the selected LR weights and the chosen recording score and threshold rule. The handoff checks reloaded recording scores and decisions against the saved operating predictions; waveform parity was checked separately in R3-M2.


In [ ]:
if not HANDOFF.exists():
    run_command([ROOT/'scripts/r3_n1_lr_bundle.py', '--evaluation-dir', OPERATING, '--fit-dir', FIT, '--output', HANDOFF])
handoff = read_json(HANDOFF/'evidence.json')
assert handoff['task'] == 'R3-N1-LR bundle handoff' and handoff['status'] == 'development_handoff_passed'
assert handoff['source_evaluation_sha256'] == digest(OPERATING/'evidence.json')
assert handoff['source_fit_sha256'] == digest(FIT/'temperature.json')
assert handoff['bundle_json_sha256'] == digest(HANDOFF/'bundle/bundle.json')
sys.path.insert(0, str(ROOT/'src'))
from echoproof.bundle import load_bundle
load_bundle(HANDOFF/'bundle')
print(json.dumps(handoff, indent=2))


Share the complete `runs/r3/r3_n1_lr/selected_lr_v1/` directory for review. It contains validation predictions, fitted temperature, operating evidence and the development bundle. No official promotion, lock update, test scoring or model retraining is performed.
